# Reduce dati schede
Keep only records whose IDs occur in `png-manifest.json`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
 
folder = Path('.')
manifest_path = folder / 'png-manifest.json'
csv_path = folder / 'data/dati_schede.csv'
geojson_path = folder / 'data/lat_long.json'

ModuleNotFoundError: No module named 'pandas'

In [ ]:
with manifest_path.open(encoding='utf-8') as f:
    manifest_ids = {str(item).strip() for item in json.load(f)}

dati = pd.read_csv(csv_path, dtype={'id_scheda_originale': 'string'}, low_memory=False)
dati_reduced = dati[dati['id_scheda_originale'].str.strip().isin(manifest_ids)].copy()
dati_reduced.to_csv(folder / 'dati_schede_reduced.csv', index=False)

with geojson_path.open(encoding='utf-8') as f:
    geojson = json.load(f)

features_reduced = [
    feature for feature in geojson['features']
    if str(feature.get('properties', {}).get('ID', '')).strip() in manifest_ids
]
geojson_reduced = {**geojson, 'features': features_reduced}

with (folder / 'lat_long_reduced.json').open('w', encoding='utf-8') as f:
    json.dump(geojson_reduced, f, ensure_ascii=False, indent=2)

print(f'Manifest IDs: {len(manifest_ids)}')
print(f'CSV rows kept: {len(dati_reduced)} / {len(dati)}')
print(f'GeoJSON features kept: {len(features_reduced)} / {len(geojson["features"])}')